# File and Raw data observations

## Assumptions

## Ideas / park for later

# Reading in the data

In [ ]:
#import packages needed
import sys
from pathlib import Path

for candidate in (Path.cwd(), Path.cwd().parent):
    if (candidate / "src").is_dir():
        sys.path.insert(0, str(candidate / "src"))
        break

import duckdb
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from stdnum.gb import nhs
from data_profile import profile_dataframe, inspect_column, parse_messy_dates


In [ ]:
#check how filepaths are resolving
from pathlib import Path

print("cwd:", Path.cwd())
data_dir = Path("../data")
print("resolved path:", data_dir.resolve())
print("exists:", data_dir.exists())
print("csvs found:", list(data_dir.glob("*.csv"))[:3])


In [ ]:
#Read the csv
df = pd.read_csv("../data/[filename].csv")
df.head()

In [ ]:
#iterate to load all files in a specific folder
from pathlib import Path

data_dir = Path("../data")
files = {}
for path in sorted(data_dir.glob("*.csv")):
    print(path)
    df_i = pd.read_csv(path)
    df_i["source_file"] = path.stem
    files[path.stem] = df_i
list(files.keys())


In [ ]:
#Read in excel
df = pd.read_excel(path)          # single sheet
xls = pd.ExcelFile(path)          # if multiple sheets
xls.sheet_names
df = pd.read_excel(path, sheet_name="Sheet1")

#preview raw rows
raw = pd.read_excel(path, header=None, nrows=10)
raw
#skip N leading rows
df = pd.read_excel(path, skiprows=3)

#skip leading and footer rows
df = pd.read_excel(path, skiprows=3, skipfooter=2)

#flatten merged rows
df = pd.read_excel(path, header=[0, 1])
df.columns = [" ".join(str(c) for c in col if "Unnamed" not in str(c)).strip()
              for col in df.columns]


In [ ]:
#if it makes sense, combine files into one df
selected = ["file1", "file2"]  # names come from files.keys() above
combined = pd.concat([files[k] for k in selected], ignore_index=True)


In [ ]:
#See the list of column headers
#print(df.columns.tolist())
#df=files[name] #inspect a file if more than one
df.columns.tolist()

# Data Quality Checks and cleaning

In [ ]:
#profile the data for single file
pd.set_option("display.max_colwidth", 200) #set the max columnwidth
pd.set_option("display.max_rows", None)   # show all columns of the profiled dataframe, don't truncate to 60
profile = profile_dataframe(df)
print("shape:", profile["shape"], "| duplicate rows:", profile["duplicate_rows"])
profile["columns"]


In [ ]:
#profiling data for multiple files - Can be very long
pd.set_option("display.max_colwidth", 200)  # set the max column width
pd.set_option("display.max_rows", None)     # show all columns of the profiled dataframe, don't truncate to 60

for name, df_i in files.items():
    print(f"=== {name} ===")
    print(df_i.columns.tolist())
    profile = profile_dataframe(df_i)
    print("shape:", profile["shape"], "| duplicate rows:", profile["duplicate_rows"])
    display(profile["columns"])


In [ ]:
#routines to review one column
#df["column_name"].unique() #list of unique values
#df["column_name"].value_counts() # list of values with count 
inspect_column(df, "column_name") #hist of values

In [ ]:
#Remove rows if applicable
#df = df[df["Period"] != "TOTAL"]

In [ ]:
# convert dates to actual dates.
date_cols = ["date_col1", "date_col2"]  # replace with the actual date column names

for col in date_cols:
    df[f"{col}_parsed"] = parse_messy_dates(df[col])
    n_failed = (df[f"{col}_parsed"].isna() & df[col].notna()).sum()
    print(f"{col}: {n_failed} values failed to parse out of {df[col].notna().sum()} non-null")


In [ ]:
# convert date columns to dates for multiple files
date_cols_by_file = {
    # "file1": ["date_col1", "date_col2"],
    # add an entry per file, listing only the columns you want parsed
}

for name, df_i in files.items():
    print(f"=== {name} ===")
    for col in date_cols_by_file.get(name, []):
        df_i[f"{col}_parsed"] = parse_messy_dates(df_i[col])
        n_failed = (df_i[f"{col}_parsed"].isna() & df_i[col].notna()).sum()
        print(f"{col}: {n_failed} values failed to parse out of {df_i[col].notna().sum()} non-null")


In [ ]:
#Figure out the primary key / unique identifier from the data profile and confirm it is unique
#int(df.duplicated(subset=["col1", "col2"]).sum())
df = files["file1"]
int(df.duplicated(subset=["id_col", "date_col"]).sum())


In [ ]:
# Check if zero means zero or no data - infer if some rows need to be removed due to no data

# Design datamart to answer question(s)

In [ ]:
con = duckdb.connect()  # in-memory; use duckdb.connect("database.duckdb") for a file that persists across kernel restarts
con.register("rawdata", df)  # makes your cleaned pandas df queryable as a duckdb table named "rawdata"
con.sql("SELECT * FROM rawdata LIMIT 5").df()


In [ ]:
#create SQL datamart for answering the question
con.sql("""
    CREATE OR REPLACE TABLE datamart AS
    SELECT "col a" as ColA,
    "col b" as ColB,
    "col c" as ColC,
    date_col,
    "measure col" as Measure
    FROM rawdata
""")
con.sql("SELECT COUNT(*) FROM datamart").df()


# Visualise / Analyze

In [ ]:
#Sample bar chart
result = con.sql("SELECT ColA, SUM(Measure) AS total FROM datamart GROUP BY ColA ORDER BY total DESC LIMIT 10").df()
result.plot(x="ColA", y="total", kind="bar")
plt.tight_layout()


In [ ]:
#Sample line chart
result = con.sql("SELECT date_col, SUM(Measure) AS total FROM datamart GROUP BY date_col ORDER BY date_col").df()
result.plot(x="date_col", y="total", kind="line")
plt.tight_layout()


In [ ]:
#Sample line chart with multiple lines
result = con.sql("""
            SELECT date_col, value_col, category_col
            FROM rawdata
            WHERE some_column = 'some_value'
            """).df()
#result
result.pivot(index="date_col", columns="category_col", values="value_col").plot(kind="line")
plt.tight_layout()
#result.groupby("date_col")["value_col"].sum().plot(kind="line")

In [ ]:
## Python help functions

# ? and ?? — inline docs
pd.read_csv?          # docstring, signature, defaults
pd.read_csv??         # actual source code
df.groupby?

# Tab completion
df.<Tab>              # list all methods/attributes on df
pd.read_<Tab>         # list matching functions

# help() — plain Python docs, works outside Jupyter too
help(pd.DataFrame.merge)

# dir() — list what's available when you forget the method name
dir(df)
[m for m in dir(df) if "date" in m.lower()]   # filter by keyword

# Shift+Tab — cursor inside function parens, shows signature/docstring inline

df.shape                    # (rows, cols)
df.head()                   # the actual values
df.dtypes                   # column types
df.columns.tolist()         # list of columns
df["col"].value_counts()    # list of values in a column
df.isna().sum()             # did the change introduce/fix nulls?
